# Smart Lab - reusable-bin detector training (Colab GPU)
Run **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.
Reads `MyDrive/SmartLab/bin_dataset.zip` (made by `tools/vision_train`), trains YOLO11n and YOLO26n, tests them on the
unseen scenes + no-bin pictures, and writes everything to `MyDrive/SmartLab/runs/<date>/` - Claude fetches it from there.
No photo of people is in the dataset; nothing else from the lab is uploaded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install ultralytics==8.4.* onnx onnxruntime
import datetime, os, shutil, zipfile
ROOT = '/content/drive/MyDrive/SmartLab'
OUT = f"{ROOT}/runs/{datetime.datetime.now():%Y-%m-%d_%H%M}"
os.makedirs(OUT, exist_ok=True)
zipfile.ZipFile(f'{ROOT}/bin_dataset.zip').extractall('/content/data')
print(os.listdir('/content/data'))

In [ ]:
from ultralytics import YOLO
res = {}
for name, w in [('y11', 'yolo11n.pt'), ('y26', 'yolo26n.pt')]:
    YOLO(w).train(data='/content/data/data.yaml', epochs=120, imgsz=640, batch=32, device=0, project='/content/runs',
                  name=name, exist_ok=True, seed=0, patience=30, fliplr=0.5, degrees=10, scale=0.6, mosaic=1.0, hsv_v=0.5)
    res[name] = f'/content/runs/{name}/weights/best.pt'

In [ ]:
import glob, json
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - ix*iy
    return ix*iy/u if u else 0
T = '/content/data/images/test'
POS = [p for p in glob.glob(T + '/*.jpg') if not os.path.basename(p).startswith('neg_')]
NEG = glob.glob(T + '/neg_*') + glob.glob('/content/data/negtest/*.jpg')
report = {}
for name, w in res.items():
    m = YOLO(w); top = lambda p: (lambda r: (float(r.boxes.conf[0]), r.boxes.xyxyn[0].tolist()) if len(r.boxes) else (0.0, None))(m.predict(p, conf=0.05, verbose=False)[0])
    pos = []
    for p in POS:
        cx, cy, bw, bh = map(float, open(p.replace('images', 'labels')[:-4] + '.txt').read().split()[1:])
        pos.append((top(p), (cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2)))
    neg = [top(p)[0] for p in NEG]
    report[name] = {str(t): {'found': sum(1 for (c, b), g in pos if c >= t and b and iou(b, g) >= .5), 'of': len(POS),
                            'false_bins': sum(c >= t for c in neg), 'of_neg': len(NEG)} for t in (.4, .5, .6, .7)}
print(json.dumps(report, indent=1)); json.dump(report, open(f'{OUT}/test_report.json', 'w'), indent=1)

In [ ]:
for name, w in res.items():
    onnx = YOLO(w).export(format='onnx', imgsz=640, opset=12, simplify=False, dynamic=False)
    shutil.copytree(f'/content/runs/{name}', f'{OUT}/{name}', dirs_exist_ok=True)
print('saved to', OUT)